In [ ]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [ ]:
!pip -q install configparser langchain openai
!pip -q install transformers huggingface_hub 
!pip -q install chromadb

In [ ]:
from langchain.llms.base import LLM
from langchain.llms.utils import enforce_stop_tokens
from langchain.llms import OpenAI
from langchain.tools import Tool
from langchain.agents import initialize_agent

In [ ]:
#Then convert the transformers pipeline into fact_llm 
import os
import dotenv
dotenv.load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
fact_llm = OpenAI(temperature=0.1)

In [ ]:
from langchain import PromptTemplate, HuggingFaceHub, LLMChain

template = """Question: {question}

Answer: Let's think step by step."""

prompt_open = PromptTemplate(template=template, input_variables=["question"])

open_chain = LLMChain(prompt=prompt_open,llm = fact_llm)

In [ ]:
question = "What NFL team won the Super Bowl in the year Justin Beiber was born?"

print(open_chain.run(question))

In [ ]:
from langchain.chains import RetrievalQA
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
recurSplitter = RecursiveCharacterTextSplitter(chunk_size=100,
                                               chunk_overlap=20,
                                               length_function=len)

In [ ]:
with open('vim_play.txt') as lin:
  txt_lin = lin.read()

linux_docs = recurSplitter.create_documents([txt_lin])

In [ ]:
with open('space_shortened.csv') as spc:
  txt_spc = spc.read()

space_docs = recurSplitter.create_documents([txt_spc])

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()

In [ ]:
from langchain.vectorstores import FAISS
space_chroma = FAISS.from_documents(space_docs, embeddings)

In [ ]:
lin_chroma = FAISS.from_documents(linux_docs,embeddings)

In [ ]:
# Starting the custom tool route

In [ ]:
lin_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=lin_chroma.as_retriever())

In [ ]:
space_retriever =RetrievalQA.from_chain_type(llm=fact_llm, 
                                           chain_type="stuff", 
                                           retriever=space_chroma.as_retriever())

In [ ]:
tools = [
    Tool(
        name = "Space Farers",
        func=space_retriever.run,
        description="useful for when you need to answer questions about the space farers."
    ),
    Tool(
        name = "Linux Commands",
        func=lin_retriever.run,
        description="useful for when you need to answer questions about Linux commands."
    ),
]

In [ ]:
example_agent = initialize_agent(tools, 
                         fact_llm, 
                         agent="zero-shot-react-description", 
                         verbose=True)

In [ ]:
example_agent.run("How many space farers are there?")

In [ ]:
# Starting the custom vectorstore agent route

In [ ]:
from langchain.agents.agent_toolkits import (
    create_vectorstore_router_agent,
    VectorStoreRouterToolkit,
    VectorStoreInfo,
)

In [ ]:
space_vectorstore_info = VectorStoreInfo(
    name="space",
    description="Used when need to answer about Space fares",
    vectorstore=space_chroma
)

In [ ]:
linux_vectorstore_info = VectorStoreInfo(
    name="linux",
    description="Used when need to answer about playbook",
    vectorstore=lin_chroma
)

In [ ]:
router_toolkit = VectorStoreRouterToolkit(
    vectorstores=[space_vectorstore_info,
                  linux_vectorstore_info],
    llm=fact_llm
)
agent_executor = create_vectorstore_router_agent(
    llm=fact_llm,
    toolkit=router_toolkit,
    verbose=True
)

In [ ]:
query = """How many Space fares are there?"""

In [ ]:
agent_executor.run(query)

In [ ]:
agent_executor.save_agent("multi_doc_agent.json")